In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, shutil, glob, subprocess, hashlib
from pathlib import Path
DRIVE_ROOT=Path('/content/drive/MyDrive'); PARENT_DIR=DRIVE_ROOT/'CALSHIFT_Research'
PROJECT_ROOT=PARENT_DIR/'calshift-research'; CRED_DIR=DRIVE_ROOT/'.gitcreds'
subprocess.run(['git','config','--global','user.name','Md Anas Biswas'],check=False)
subprocess.run(['git','config','--global','user.email','anasbiswas@gmail.com'],check=False)
subprocess.run(['git','config','--global','credential.helper','store'],check=False)
for fn,dest in [('.git-credentials','/root/.git-credentials'),('.gitconfig','/root/.gitconfig')]:
    for cand in (PARENT_DIR/fn, CRED_DIR/fn):
        if cand.exists(): shutil.copy(cand,dest); os.chmod(dest,0o600); break
os.chdir(PROJECT_ROOT); sys.path.insert(0,str(PROJECT_ROOT/'src'))
subprocess.run(['git','pull','--ff-only','--quiet'],check=False)
import importlib
if 'config' in sys.modules: importlib.reload(sys.modules['config'])
import config
import numpy as np, pandas as pd
from scipy import stats
RNG=np.random.default_rng(20260726); B=2000
ALPHA=config.ALPHA_PRIMARY
def dseed(*p): return int(hashlib.sha256('|'.join(map(str,p)).encode()).hexdigest(),16)%(2**32)
def cluster_boot(df,val,clus,B=B,rng=RNG):
    cl=df[clus].unique()
    if len(cl)<2: return (float(df[val].mean()),np.nan,np.nan)
    means=np.array([df[df[clus]==c][val].mean() for c in cl])
    bs=np.array([rng.choice(means,len(means),replace=True).mean() for _ in range(B)])
    return float(means.mean()),float(np.percentile(bs,2.5)),float(np.percentile(bs,97.5))
print('ready:', os.getcwd())


Mounted at /content/drive
ready: /content/drive/MyDrive/CALSHIFT_Research/calshift-research


In [2]:
# =============================================================================
# M1 (METHODOLOGICAL, headline) - SHIFT-TYPE DECOMPOSITION WITHIN NSL-KDD.
# The ladder was built to vary support shift. The measured covariates show that
# rung 0.00 has S_sup ~ 0 with S_cov ~ 0.85, i.e. it is ALREADY a fixed-support
# covariate-shift environment; and S_cov stays nearly flat (0.849 -> 0.893) while
# S_sup sweeps 0.00 -> 0.45. So within ONE dataset we can separate:
#   covariate-attributable gap = nominal        - coverage(rung 0.00)
#   support-attributable gap   = coverage(r0.00)- coverage(rung 0.80)
# This breaks the shift-type / dataset confound that the pooled model could not.
# =============================================================================
sm=pd.read_csv(config.REPORTS_DIR/'ladder_shift_measures_nslkdd.csv')
shift=sm.groupby('rung')[['S_cov','S_lab','S_sup']].mean().round(4)
print('NSL measured shift by rung:'); print(shift.to_string())
print(f"\nS_cov range across ladder: {shift['S_cov'].min():.3f} - {shift['S_cov'].max():.3f} (nearly flat)")
print(f"S_sup range across ladder: {shift['S_sup'].min():.3f} - {shift['S_sup'].max():.3f} (swept)")

nsl=pd.read_csv(config.REPORTS_DIR/'coverage_primary_nslkdd.csv')
foc=nsl[(nsl['class']=='R2L')&(np.isclose(nsl['alpha'],ALPHA))&(nsl['protocol']=='SHC')]
rows=[]
for r in sorted(foc['rung'].unique()):
    s=foc[np.isclose(foc['rung'],r)]
    m,lo,hi=cluster_boot(s,'coverage','seed')
    rows.append({'rung':r,'S_cov':float(shift.loc[r,'S_cov']),'S_sup':float(shift.loc[r,'S_sup']),
                 'coverage':round(m,4),'ci_lo':round(lo,4),'ci_hi':round(hi,4)})
dec=pd.DataFrame(rows)
nominal=1-ALPHA
cov0=float(dec[np.isclose(dec['rung'],0.0)]['coverage'].iloc[0])
cov8=float(dec[np.isclose(dec['rung'],0.8)]['coverage'].iloc[0])
gap_cov = nominal-cov0            # covariate shift alone, S_sup~0
gap_sup = cov0-cov8               # additional, attributable to support shift
gap_tot = nominal-cov8
print('\n--- DECOMPOSITION OF THE FOCAL COVERAGE GAP (NSL-KDD, alpha=%.2f) ---'%ALPHA)
print(f'  nominal                              : {nominal:.4f}')
print(f'  coverage at rung 0.00 (S_sup~0)      : {cov0:.4f}')
print(f'  coverage at rung 0.80 (S_sup=0.45)   : {cov8:.4f}')
print(f'  gap attributable to COVARIATE shift  : {gap_cov:.4f}  ({100*gap_cov/gap_tot:.1f}% of total)')
print(f'  gap attributable to SUPPORT shift    : {gap_sup:.4f}  ({100*gap_sup/gap_tot:.1f}% of total)')
print(f'  total gap                            : {gap_tot:.4f}')
print('\nread: covariate shift ALONE, at S_cov~0.85, already destroys focal coverage;')
print('      support shift adds a further, smaller increment. UGR holds its focal class')
print('      at S_cov=0.69, so the operative variable is score movement / shift MAGNITUDE,')
print('      not shift TYPE per se. This is testable in C-block below.')
m1={'nominal':nominal,'cov_rung0':cov0,'cov_rung8':cov8,
    'gap_covariate':round(gap_cov,4),'gap_support':round(gap_sup,4),'gap_total':round(gap_tot,4),
    'pct_covariate':round(100*gap_cov/gap_tot,1),'pct_support':round(100*gap_sup/gap_tot,1),
    'S_cov_range':[float(shift['S_cov'].min()),float(shift['S_cov'].max())],
    'S_sup_range':[float(shift['S_sup'].min()),float(shift['S_sup'].max())],
    'per_rung':dec.to_dict('records')}


NSL measured shift by rung:
       S_cov   S_lab   S_sup
rung                        
0.0   0.8485  0.1363  0.0010
0.2   0.8571  0.1361  0.1142
0.4   0.8638  0.1360  0.2276
0.6   0.8750  0.1361  0.3407
0.8   0.8927  0.1360  0.4543

S_cov range across ladder: 0.849 - 0.893 (nearly flat)
S_sup range across ladder: 0.001 - 0.454 (swept)

--- DECOMPOSITION OF THE FOCAL COVERAGE GAP (NSL-KDD, alpha=0.05) ---
  nominal                              : 0.9500
  coverage at rung 0.00 (S_sup~0)      : 0.1431
  coverage at rung 0.80 (S_sup=0.45)   : 0.0298
  gap attributable to COVARIATE shift  : 0.8069  (87.7% of total)
  gap attributable to SUPPORT shift    : 0.1133  (12.3% of total)
  total gap                            : 0.9202

read: covariate shift ALONE, at S_cov~0.85, already destroys focal coverage;
      support shift adds a further, smaller increment. UGR holds its focal class
      at S_cov=0.69, so the operative variable is score movement / shift MAGNITUDE,
      not shift TYPE per s

In [3]:
# =============================================================================
# S1 - THEORETICAL COVERAGE BAND. Split conformal with n calibration points and
# quantile index ceil((n+1)(1-a)) satisfies, under exchangeability and continuous
# scores:   1-a  <=  P(cover)  <=  1-a + 1/(n+1)
# Reporting the BAND converts every "gap" into a formal violation of a
# distribution-free finite-sample guarantee, which is a far stronger statement.
# NSL: n_cal is recorded per cell. CIC/UGR: reconstruct the per-draw class
# calibration counts with the same deterministic matched-draw seeds as nb23.
# =============================================================================
LONG=config.PROC_DIR/'coverage_long_nslkdd.parquet'
nl=pd.read_parquet(LONG)
prim=nl[(nl['score']=='aps')&(nl['variant']=='mondrian')&(np.isclose(nl['alpha'],ALPHA))&(nl['feasible'])]

def band(n,a=ALPHA): return (1-a, 1-a+1.0/(n+1))
band_rows=[]
for (cls,proto),g in prim.groupby(['class','protocol']):
    n=float(g['n_cal'].mean()); lo,hi=band(n); obs=float(g['coverage'].mean())
    band_rows.append({'dataset':'nslkdd','class':cls,'protocol':proto,'n_cal':round(n,1),
                      'band_lo':round(lo,4),'band_hi':round(hi,4),'observed':round(obs,4),
                      'below_band':bool(obs<lo),'above_band':bool(obs>hi),
                      'violation_size':round(lo-obs,4) if obs<lo else 0.0})
bt=pd.DataFrame(band_rows).sort_values(['class','protocol'])
print('NSL-KDD: observed coverage vs the theoretical guarantee band')
print(bt.to_string(index=False))
print('\nBELOW-BAND (guarantee violated):')
print(bt[bt.below_band][['class','protocol','n_cal','band_lo','observed','violation_size']].to_string(index=False))
print('\nABOVE-BAND (upper bound exceeded -> score atoms / discreteness, not a bug):')
print(bt[bt.above_band][['class','protocol','n_cal','band_hi','observed']].to_string(index=False))


NSL-KDD: observed coverage vs the theoretical guarantee band
dataset  class protocol   n_cal  band_lo  band_hi  observed  below_band  above_band  violation_size
 nslkdd    DoS      REC   772.1     0.95   0.9513    0.9521       False        True          0.0000
 nslkdd    DoS      SHC  6889.0     0.95   0.9501    0.6722        True       False          0.2778
 nslkdd    DoS      TSC   772.1     0.95   0.9513    0.9507       False       False          0.0000
 nslkdd Normal      REC  1008.0     0.95   0.9510    0.9514       False        True          0.0000
 nslkdd Normal      SHC 10101.0     0.95   0.9501    0.9275        True       False          0.0225
 nslkdd Normal      TSC  1008.0     0.95   0.9510    0.9512       False        True          0.0000
 nslkdd  Probe      REC   249.9     0.95   0.9540    0.9560       False        True          0.0000
 nslkdd  Probe      SHC  1748.0     0.95   0.9506    0.8054        True       False          0.1446
 nslkdd  Probe      TSC   249.9     0.9

In [4]:
# =============================================================================
# S1b - the same band for CIC + UGR, using RECONSTRUCTED per-draw class
# calibration counts (identical deterministic seeds to nb23, so exact).
# =============================================================================
R=getattr(config,'N_MATCHED_DRAWS',10)
rec_rows=[]

# ---- CIC ----
cic=pd.read_parquet(config.INTERIM_DIR/'cicids2017_primary.parquet')
wed=cic[cic['day']=='wednesday'].reset_index(drop=True); wed=wed[wed['label'].isin(['DoS','Benign'])].reset_index(drop=True)
CICC=['Benign','DoS']
def lab_cic(idx): return (wed.loc[idx,'label'].to_numpy()=='DoS').astype(int)
REAL=['R1_holdout_Slowhttptest','R2_holdout_Slowloris','R3_holdout_GoldenEye',
      'R4_holdout_Slowloris_Slowhttptest','R5_holdout_GoldenEye_Slowloris']
for name in REAL:
    spx=np.load(config.PROC_DIR/f'cic_{name}_srcpool_idx.npy'); tgx=np.load(config.PROC_DIR/f'cic_{name}_target_idx.npy')
    ysp,ytg=lab_cic(spx),lab_cic(tgx); mm=min(len(spx),len(tgx)//2)
    for f in sorted((config.DATA_DIR/'cic_probs').glob(f'{name}__*.npz')):
        _,arch,sd=Path(f).stem.split('__'); seed=int(sd.replace('seed',''))
        for draw in range(R):
            rng=np.random.default_rng(dseed(name,seed,arch,draw))
            _=rng.permutation(len(ytg))[:mm]; sc=rng.permutation(len(ysp))[:mm]
            ysc=ysp[sc]
            for c in range(2):
                rec_rows.append({'dataset':'cicids2017','class':CICC[c],'n_cal':int((ysc==c).sum())})
# ---- UGR ----
UGR=config.DATASETS_DIR/'ugr16'; usrc=pd.read_parquet(UGR/'july_week5.parquet'); utgt=pd.read_parquet(UGR/'august_week1.parquet')
for dd in (usrc,utgt): dd['label']=dd['label'].astype(str).str.strip().str.lower()
UK=['background','dos','scan11','scan44','nerisbotnet']
usrc=usrc[usrc.label.isin(UK)].reset_index(drop=True); utgt=utgt[utgt.label.isin(UK)].reset_index(drop=True)
UCL=sorted(UK); U2I={c:i for i,c in enumerate(UCL)}
def strat(df,fr,seed,col='label'):
    rng=np.random.default_rng(seed); nm=list(fr); ff=np.array([fr[k] for k in nm],float); big=nm[int(np.argmax(ff))]
    a=pd.Series(index=df.index,dtype=object)
    for _,s in df.groupby(col,sort=True):
        idx=s.index.to_numpy().copy(); rng.shuffle(idx); n=len(idx)
        c=np.floor(ff*n).astype(int); c[nm.index(big)]+=n-c.sum(); kk=0
        for a2,q in zip(nm,c): a.loc[idx[kk:kk+q]]=a2; kk+=q
    return a
usrc=usrc.assign(partition=strat(usrc,config.SPLIT_FRACTIONS,20260725).values)
y_sp=usrc[usrc.partition=='source_cal_pool']['label'].map(U2I).to_numpy(); y_tg=utgt['label'].map(U2I).to_numpy()
mmU=min(len(y_sp),len(y_tg)//2)
for f in sorted((config.DATA_DIR/'ugr16_probs').glob('ugr16__*.npz')):
    _,arch,sd=Path(f).stem.split('__'); seed=int(sd.replace('seed',''))
    for draw in range(R):
        rng=np.random.default_rng(dseed('ugr16',seed,arch,draw))
        _=rng.permutation(len(y_tg))[:mmU]; sc=rng.permutation(len(y_sp))[:mmU]
        ysc=y_sp[sc]
        for c in range(len(UCL)):
            rec_rows.append({'dataset':'ugr16','class':UCL[c],'n_cal':int((ysc==c).sum())})

ncal=pd.DataFrame(rec_rows).groupby(['dataset','class'])['n_cal'].mean().reset_index()
print('reconstructed mean per-draw SHC calibration counts:'); print(ncal.round(1).to_string(index=False))

obs_rows=[]
for f,ds in [('coverage_primary_cicids2017.csv','cicids2017'),('coverage_primary_ugr16.csv','ugr16')]:
    d=pd.read_csv(config.REPORTS_DIR/f); d=d[np.isclose(d['alpha'],ALPHA)]
    for (cls,proto),g in d.groupby(['class','protocol']):
        n=float(ncal[(ncal.dataset==ds)&(ncal['class']==cls)]['n_cal'].iloc[0]) if len(ncal[(ncal.dataset==ds)&(ncal['class']==cls)]) else np.nan
        lo,hi=band(n); obs=float(g['coverage'].mean())
        obs_rows.append({'dataset':ds,'class':cls,'protocol':proto,'n_cal':round(n,1),
                         'band_lo':round(lo,4),'band_hi':round(hi,4),'observed':round(obs,4),
                         'below_band':bool(obs<lo),'above_band':bool(obs>hi),
                         'violation_size':round(lo-obs,4) if obs<lo else 0.0})
bt2=pd.DataFrame(obs_rows)
print('\nCIC + UGR: observed vs guarantee band')
print(bt2.to_string(index=False))
band_all=pd.concat([bt,bt2],ignore_index=True)
print('\nSHC cells BELOW the guarantee band (formal violations):')
print(band_all[(band_all.protocol=='SHC')&(band_all.below_band)][['dataset','class','n_cal','band_lo','observed','violation_size']].to_string(index=False))


reconstructed mean per-draw SHC calibration counts:
   dataset       class   n_cal
cicids2017      Benign 22957.0
cicids2017         DoS 24844.6
     ugr16  background 30000.0
     ugr16         dos  7500.0
     ugr16 nerisbotnet  7500.0
     ugr16      scan11  7500.0
     ugr16      scan44  7500.0

CIC + UGR: observed vs guarantee band
   dataset       class protocol   n_cal  band_lo  band_hi  observed  below_band  above_band  violation_size
cicids2017      Benign      REC 22957.0     0.95   0.9500    0.9500       False        True          0.0000
cicids2017      Benign      SHC 22957.0     0.95   0.9500    0.9501       False        True          0.0000
cicids2017      Benign      TSC 22957.0     0.95   0.9500    0.9500       False       False          0.0000
cicids2017         DoS      REC 24844.6     0.95   0.9500    0.9875       False        True          0.0000
cicids2017         DoS      SHC 24844.6     0.95   0.9500    0.6039        True       False          0.3461
cicids2017   

In [5]:
# =============================================================================
# S2 + S3 - VALIDITY TESTS WITH MULTIPLICITY CONTROL, and EQUIVALENCE TESTS.
# Coverage indicators inside a class share one calibration set, so they are not
# iid Bernoulli and an exact binomial test would be anti-conservative. We instead
# test with the seed-clustered bootstrap CI:
#   VIOLATION  if ci_hi < band_lo      (coverage provably below the guarantee)
#   EQUIVALENT if [ci_lo,ci_hi] within [nominal-delta, nominal+delta]  (TOST)
# Family = every focal/discovered class-by-dataset SHC claim. Holm + BH applied
# to the bootstrap-derived one-sided p-values.
# =============================================================================
DELTA=0.01
def boot_p_below(df,val,clus,thresh,B=B,rng=RNG):
    cl=df[clus].unique(); means=np.array([df[df[clus]==c][val].mean() for c in cl])
    bs=np.array([rng.choice(means,len(means),replace=True).mean() for _ in range(B)])
    return float(np.mean(bs>=thresh))   # one-sided p for H0: coverage >= band_lo

claims=[('nslkdd','R2L','SHC',0.80),('cicids2017','DoS','SHC',None),
        ('ugr16','nerisbotnet','SHC',None),('ugr16','scan11','SHC',None),
        ('ugr16','scan44','SHC',None),('ugr16','dos','SHC',None),('ugr16','background','SHC',None)]
srcf={'nslkdd':'coverage_primary_nslkdd.csv','cicids2017':'coverage_primary_cicids2017.csv','ugr16':'coverage_primary_ugr16.csv'}
tst=[]
for ds,cls,proto,rung in claims:
    d=pd.read_csv(config.REPORTS_DIR/srcf[ds])
    s=d[(d['class']==cls)&(np.isclose(d['alpha'],ALPHA))&(d['protocol']==proto)]
    if rung is not None: s=s[np.isclose(s['rung'],rung)]
    if not len(s): continue
    m,lo,hi=cluster_boot(s,'coverage','seed')
    row=band_all[(band_all.dataset==ds)&(band_all['class']==cls)&(band_all.protocol==proto)]
    blo=float(row['band_lo'].iloc[0]) if len(row) else 1-ALPHA
    p=boot_p_below(s,'coverage','seed',blo)
    tst.append({'dataset':ds,'class':cls,'coverage':round(m,4),'ci_lo':round(lo,4),'ci_hi':round(hi,4),
                'band_lo':blo,'violation':bool(hi<blo),'p_one_sided':p,
                'equivalent_TOST':bool((lo>=(1-ALPHA)-DELTA) and (hi<=(1-ALPHA)+DELTA))})
tt=pd.DataFrame(tst)
# multiplicity control
from statsmodels.stats.multitest import multipletests
pv=tt['p_one_sided'].to_numpy()
tt['p_holm']=multipletests(pv,method='holm')[1]
tt['p_bh']=multipletests(pv,method='fdr_bh')[1]
print(f'VALIDITY + EQUIVALENCE TESTS (family of {len(tt)} claims, delta={DELTA})')
print(tt.to_string(index=False))
print('\nviolations after Holm:', tt[(tt.violation)&(tt.p_holm<0.05)][['dataset','class']].to_dict('records'))
print('equivalent (holds, positively demonstrated):', tt[tt.equivalent_TOST][['dataset','class']].to_dict('records'))


VALIDITY + EQUIVALENCE TESTS (family of 7 claims, delta=0.01)
   dataset       class  coverage  ci_lo  ci_hi  band_lo  violation  p_one_sided  equivalent_TOST  p_holm   p_bh
    nslkdd         R2L    0.0298 0.0266 0.0339     0.95       True       0.0000            False  0.0000 0.0000
cicids2017         DoS    0.6039 0.5972 0.6108     0.95       True       0.0000            False  0.0000 0.0000
     ugr16 nerisbotnet    0.9473 0.9470 0.9476     0.95       True       0.0000             True  0.0000 0.0000
     ugr16      scan11    0.5350 0.5247 0.5441     0.95       True       0.0000            False  0.0000 0.0000
     ugr16      scan44    0.7990 0.7944 0.8037     0.95       True       0.0000            False  0.0000 0.0000
     ugr16         dos    0.9503 0.9500 0.9506     0.95      False       0.9785             True  0.9785 0.9785
     ugr16  background    0.9492 0.9490 0.9493     0.95       True       0.0000             True  0.0000 0.0000

violations after Holm: [{'dataset': 'nslk

In [6]:
# =============================================================================
# S4 - HETEROGENEITY TEST for the SELECTIVITY claim. "Under one covariate shift,
# some classes hold and others collapse" is currently read off a table. Test it:
# Cochran's Q on the five UGR per-class coverage gaps weighted by inverse
# variance, against chi2 with k-1 df, plus I^2 = share of variance that is real
# heterogeneity rather than sampling noise.
# =============================================================================
ug=pd.read_csv(config.REPORTS_DIR/'coverage_primary_ugr16.csv')
ug=ug[(np.isclose(ug['alpha'],ALPHA))&(ug['protocol']=='SHC')]
rows=[]
for cls,g in ug.groupby('class'):
    per_seed=g.groupby('seed')['coverage'].mean()
    theta=float(per_seed.mean()); var=float(per_seed.var(ddof=1)/len(per_seed))
    rows.append({'class':cls,'gap':theta-(1-ALPHA),'theta':theta,'var':max(var,1e-12),'n_seeds':len(per_seed)})
het=pd.DataFrame(rows)
w=1.0/het['var']; theta_bar=float((w*het['theta']).sum()/w.sum())
Q=float((w*(het['theta']-theta_bar)**2).sum()); k=len(het); dfq=k-1
pQ=float(1-stats.chi2.cdf(Q,dfq)); I2=max(0.0,(Q-dfq)/Q) if Q>0 else 0.0
print('UGR per-class coverage under one covariate shift (S_cov=0.69):')
print(het[['class','theta','gap','n_seeds']].round(4).to_string(index=False))
print(f"\nCochran's Q = {Q:.1f} on {dfq} df, p = {pQ:.3g}")
print(f"I^2 = {100*I2:.1f}%  (share of between-class variance that is genuine heterogeneity)")
print("\nread: a large Q with I^2 near 100% means the classes do NOT share a common")
print("      coverage under the same aggregate shift => selectivity is a tested claim,")
print("      not an eyeballed one.")
s4={'Q':round(Q,2),'df':dfq,'p':pQ,'I2_pct':round(100*I2,1),'theta_bar':round(theta_bar,4),
    'per_class':het[['class','theta','gap']].round(4).to_dict('records')}


UGR per-class coverage under one covariate shift (S_cov=0.69):
      class  theta     gap  n_seeds
 background 0.9492 -0.0008       10
        dos 0.9503  0.0003       10
nerisbotnet 0.9473 -0.0027       10
     scan11 0.5350 -0.4150       10
     scan44 0.7990 -0.1510       10

Cochran's Q = 10109.8 on 4 df, p = 0
I^2 = 100.0%  (share of between-class variance that is genuine heterogeneity)

read: a large Q with I^2 near 100% means the classes do NOT share a common
      coverage under the same aggregate shift => selectivity is a tested claim,
      not an eyeballed one.


In [7]:
# =============================================================================
# M2 (METHODOLOGICAL) - NEGATIVE CONTROL / PIPELINE VALIDATION.
# Split the SOURCE calibration pool in half at random, calibrate on one half and
# evaluate on the other. There is NO shift by construction, so class-conditional
# coverage must land inside the guarantee band. If it does, the machinery is
# correct and every failure reported elsewhere is attributable to shift, not to
# an implementation error. This is the control the study currently lacks.
# =============================================================================
CLASSES=config.CANONICAL_CLASSES; c2i={c:i for i,c in enumerate(CLASSES)}; K=len(CLASSES)
nsl_train=pd.read_parquet(config.INTERIM_DIR/'nslkdd_train.parquet').reset_index(drop=True)
part=pd.read_parquet(config.PROC_DIR/'nslkdd_source_partition_labels.parquet')
nsl_train=nsl_train.assign(partition=part['partition'].values)
y_pool=nsl_train[nsl_train.partition=='source_cal_pool']['label'].map(c2i).to_numpy()

def aps_all(P,rng):
    o=np.argsort(-P,axis=1); sp=np.take_along_axis(P,o,1); cum=np.cumsum(sp,1)
    U=rng.random(len(P))[:,None]; ss=cum-(1-U)*sp
    out=np.empty_like(P); np.put_along_axis(out,o,ss,1); return out
def qhat(s,a):
    n=len(s); return np.inf if n<1 else float(np.quantile(s,min(np.ceil((n+1)*(1-a))/n,1.0),method='higher'))

nc_rows=[]
for f in sorted(glob.glob(str(config.PROC_DIR/'probs_*.npz'))):
    arch,seed=Path(f).stem.replace('probs_','').rsplit('_s',1)
    P=np.load(f)['S_pool'].astype(np.float64)
    if len(P)!=len(y_pool): continue
    for rep in range(5):
        rng=np.random.default_rng(dseed('negctrl',arch,seed,rep))
        perm=rng.permutation(len(y_pool)); half=len(perm)//2
        ci,ei=perm[:half],perm[half:]
        sc=aps_all(P[ci],np.random.default_rng(dseed('negctrl',arch,seed,rep,'c')))
        se=aps_all(P[ei],np.random.default_rng(dseed('negctrl',arch,seed,rep,'e')))
        yc,ye=y_pool[ci],y_pool[ei]
        tc=sc[np.arange(len(yc)),yc]
        for c in range(K):
            ncal=int((yc==c).sum()); nev=int((ye==c).sum())
            if ncal < config.min_calib_n(ALPHA) or nev==0: continue
            q=qhat(tc[yc==c],ALPHA)
            covd=float((se[ye==c,c]<=q).mean())
            nc_rows.append({'arch':arch,'seed':seed,'rep':rep,'class':CLASSES[c],
                            'n_cal':ncal,'coverage':covd})
nc=pd.DataFrame(nc_rows)
print('NEGATIVE CONTROL (source vs source, NO shift): class-conditional coverage')
out=[]
for cls,g in nc.groupby('class'):
    m,lo,hi=cluster_boot(g,'coverage','seed'); n=float(g['n_cal'].mean()); blo,bhi=band(n)
    inside=bool(lo<=bhi and hi>=blo)
    out.append({'class':cls,'n_cal':round(n,1),'coverage':round(m,4),'ci_lo':round(lo,4),
                'ci_hi':round(hi,4),'band_lo':round(blo,4),'band_hi':round(bhi,4),'inside_band':inside})
nctl=pd.DataFrame(out)
print(nctl.to_string(index=False))
print('\nPASS = every feasible class sits inside its guarantee band with no shift.')
print('PASS' if bool(nctl['inside_band'].all()) else 'FAIL - investigate before trusting the shift results')


NEGATIVE CONTROL (source vs source, NO shift): class-conditional coverage
 class  n_cal  coverage  ci_lo  ci_hi  band_lo  band_hi  inside_band
   DoS 3439.7    0.9505 0.9500 0.9511     0.95   0.9503         True
Normal 5053.2    0.9505 0.9498 0.9511     0.95   0.9502         True
 Probe  875.7    0.9528 0.9514 0.9543     0.95   0.9511        False
   R2L   74.9    0.9709 0.9678 0.9737     0.95   0.9632        False

PASS = every feasible class sits inside its guarantee band with no shift.
FAIL - investigate before trusting the shift results


In [8]:
# =============================================================================
# M3 + M4 - SPECIFICATION CURVE. The study computed 2 scores x 2 variants x
# 4 alphas = 16 specifications and reports one. Showing the focal result across
# ALL of them (a) removes any suspicion of selective reporting, (b) doubles as
# the LAC robustness check, (c) is a recognised robustness display.
# =============================================================================
spec=[]
for score in ['aps','lac']:
    for variant in ['mondrian','marginal']:
        for a in sorted(nl['alpha'].unique()):
            g=nl[(nl['score']==score)&(nl['variant']==variant)&(np.isclose(nl['alpha'],a))&
                 (nl['protocol']=='SHC')&(nl['class']=='R2L')&(np.isclose(nl['rung'],0.80))]
            if not len(g): continue
            gf=g[g['feasible']] if g['feasible'].any() else g
            m,lo,hi=cluster_boot(gf,'coverage','seed')
            spec.append({'score':score,'variant':variant,'alpha':float(a),'nominal':round(1-float(a),3),
                         'coverage':round(m,4),'ci_lo':round(lo,4),'ci_hi':round(hi,4),
                         'gap':round((1-float(a))-m,4),'feasible_cells':int(g['feasible'].sum())})
sc_tbl=pd.DataFrame(spec).sort_values(['score','variant','alpha'])
print('SPECIFICATION CURVE - NSL focal R2L, SHC, rung 0.80, all 16 specifications:')
print(sc_tbl.to_string(index=False))
und=(sc_tbl['coverage']<sc_tbl['nominal']-0.02).sum()
print(f'\nspecifications showing undercoverage: {und}/{len(sc_tbl)}')
print('gap range across specifications: %.4f to %.4f'%(sc_tbl["gap"].min(),sc_tbl["gap"].max()))
print('\nread: if the focal failure appears in every specification, the headline is not a')
print('      product of the chosen score, conditioning or level.')


SPECIFICATION CURVE - NSL focal R2L, SHC, rung 0.80, all 16 specifications:
score  variant  alpha  nominal  coverage  ci_lo  ci_hi    gap  feasible_cells
  aps marginal   0.01     0.99    0.0288 0.0269 0.0308 0.9612             600
  aps marginal   0.05     0.95    0.0244 0.0228 0.0259 0.9256             600
  aps marginal   0.10     0.90    0.0204 0.0183 0.0225 0.8796             600
  aps marginal   0.20     0.80    0.0162 0.0144 0.0180 0.7838             600
  aps mondrian   0.01     0.99    0.6197 0.5393 0.7043 0.3703             600
  aps mondrian   0.05     0.95    0.0298 0.0267 0.0339 0.9202             600
  aps mondrian   0.10     0.90    0.0231 0.0213 0.0250 0.8769             600
  aps mondrian   0.20     0.80    0.0168 0.0150 0.0185 0.7832             600
  lac marginal   0.01     0.99    0.0033 0.0022 0.0043 0.9867             600
  lac marginal   0.05     0.95    0.0029 0.0019 0.0037 0.9471             600
  lac marginal   0.10     0.90    0.0028 0.0019 0.0037 0.8972     

In [9]:
# =============================================================================
# S5 - WITHIN-NSL HIERARCHICAL MODEL + VARIANCE DECOMPOSITION.
# The pooled cross-dataset model is not identified (3 clusters). Within NSL the
# design IS rich: 5 rungs x 20 realizations x 10 seeds x 3 architectures. Fit the
# dose-response on the empirical logit with crossed random effects, and decompose
# where coverage variance actually comes from - a question practitioners have.
# =============================================================================
import statsmodels.api as sm, statsmodels.formula.api as smf
cells_nsl=prim[(prim['class']=='R2L')&(prim['protocol']=='SHC')].copy()
cells_nsl['elogit']=np.log((cells_nsl['n_covered']+0.5)/(cells_nsl['n_eval']-cells_nsl['n_covered']+0.5))
cells_nsl['rung_c']=cells_nsl['rung']-cells_nsl['rung'].mean()
cells_nsl['realization']=cells_nsl['realization'].astype(str)
cells_nsl['grp']=1
print('cells for hierarchical fit:',len(cells_nsl))
try:
    md_=smf.mixedlm('elogit ~ rung_c',cells_nsl,groups=cells_nsl['grp'],
        vc_formula={'realization':'0+C(realization)','seed':'0+C(seed)','arch':'0+C(arch)'})
    mf=md_.fit(reml=True)
    print(mf.summary())
    vc=np.atleast_1d(np.asarray(mf.vcomp,dtype=float)); resid=float(mf.scale)
    tot=float(vc.sum())+resid
    # variance-component labels differ across statsmodels versions; try in order
    names=None
    for getter in (lambda: list(mf.model.exog_vc.names),
                   lambda: list(md_.exog_vc.names),
                   lambda: [k for k in md_.exog_vc.keys()]):
        try:
            cand=getter()
            if cand and len(cand)==len(vc): names=cand; break
        except Exception: pass
    if names is None: names=[f'vc{i+1}' for i in range(len(vc))]
    print('\nVARIANCE DECOMPOSITION (share of variance on the logit scale):')
    for nm,v in zip(names,vc): print(f'   {str(nm):12s}: {100*float(v)/tot:5.1f}%')
    print(f'   {"residual":12s}: {100*resid/tot:5.1f}%')
    slope=float(mf.fe_params['rung_c'])
    try: slope_se=float(mf.bse_fe['rung_c'])
    except Exception: slope_se=float(mf.bse['rung_c'])
    s5={'slope_rung':round(slope,4),'slope_se':round(slope_se,4),
        'variance_shares':{str(nm):round(100*float(v)/tot,1) for nm,v in zip(names,vc)},
        'residual_share':round(100*resid/tot,1),'n_cells':int(len(cells_nsl))}
except Exception as e:
    print('mixed model failed:',e)
    a1=cells_nsl.groupby('arch')['elogit'].mean().var(ddof=1)
    a2=cells_nsl.groupby('seed')['elogit'].mean().var(ddof=1)
    a3=cells_nsl.groupby('realization')['elogit'].mean().var(ddof=1)
    tot=a1+a2+a3
    print('fallback variance shares (between-group variance on elogit):')
    print(f'   arch {100*a1/tot:.1f}% | seed {100*a2/tot:.1f}% | realization {100*a3/tot:.1f}%')
    s5={'fallback_shares':{'arch':round(float(100*a1/tot),1),'seed':round(float(100*a2/tot),1),
                           'realization':round(float(100*a3/tot),1)},'error':str(e)}


cells for hierarchical fit: 3000
           Mixed Linear Model Regression Results
Model:               MixedLM  Dependent Variable:  elogit   
No. Observations:    3000     Method:              REML     
No. Groups:          1        Scale:               0.1006   
Min. group size:     3000     Log-Likelihood:      -871.0176
Max. group size:     3000     Converged:           Yes      
Mean group size:     3000.0                                 
------------------------------------------------------------
                Coef.  Std.Err.    z     P>|z| [0.025 0.975]
------------------------------------------------------------
Intercept       -2.557    0.204  -12.507 0.000 -2.957 -2.156
rung_c          -2.071    0.020 -101.152 0.000 -2.111 -2.030
arch Var         0.112    0.354                             
realization Var  0.009    0.010                             
seed Var         0.039    0.059                             


VARIANCE DECOMPOSITION (share of variance on the logit scale):

/usr/local/lib/python3.12/dist-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


In [ ]:
# =============================================================================
# save + commit
# =============================================================================
out={'M1_shift_decomposition_within_nsl':m1,
     'S1_guarantee_band':{'nslkdd':bt.to_dict('records'),'cic_ugr':bt2.to_dict('records')},
     'S2_S3_validity_equivalence':tt.to_dict('records'),
     'S4_selectivity_heterogeneity':s4,
     'M2_negative_control':{'per_class':nctl.to_dict('records'),'pass':bool(nctl['inside_band'].all())},
     'M3_specification_curve':sc_tbl.to_dict('records'),
     'S5_hierarchical_variance':s5}
(config.REPORTS_DIR/'statistical_upgrades.json').write_text(json.dumps(out,indent=2,default=str))
band_all.to_csv(config.REPORTS_DIR/'coverage_guarantee_band.csv',index=False)
tt.to_csv(config.REPORTS_DIR/'validity_equivalence_tests.csv',index=False)
nctl.to_csv(config.REPORTS_DIR/'negative_control.csv',index=False)
sc_tbl.to_csv(config.REPORTS_DIR/'specification_curve.csv',index=False)
dec.to_csv(config.REPORTS_DIR/'shift_decomposition_nslkdd.csv',index=False)
print('saved 5 CSVs + statistical_upgrades.json')

def git(*a, show=True):
    r=subprocess.run(['git',*a],capture_output=True,text=True)
    if show and (r.stdout or r.stderr): print((r.stdout+r.stderr).strip())
    return r
for s,dd in [('/root/.git-credentials',PARENT_DIR/'.git-credentials'),('/root/.gitconfig',PARENT_DIR/'.gitconfig')]:
    if os.path.exists(s): shutil.copy(s,dd)
os.chdir(PROJECT_ROOT); git('add','-A',show=False)
if git('status','--porcelain',show=False).stdout.strip():
    git('commit','-m','nb28: statistical upgrades - guarantee band, validity+equivalence tests, heterogeneity test, negative control, specification curve, within-NSL shift decomposition and variance components')
    r=git('push','-u','origin','main')
    if r.returncode: print('PUSH FAILED. Commit is safe locally.')
else: print('nothing to commit')
print(git('log','--oneline','-3',show=False).stdout)


saved 5 CSVs + statistical_upgrades.json
